<a href="https://colab.research.google.com/github/rubenchov/Python-Remotesensing/blob/main/DownloadGEE_%5BRGB_NDVI_SAR%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
from google.colab import drive
import ee
import geemap
import pandas as pd
import numpy as np
import random
import cv2
from google.colab.patches import cv2_imshow
import os

Google Drive

In [ ]:
# Drive
drive.mount('/content/drive')
dirname = os.path.join(os.getcwd(), '/content/drive/MyDrive/2026/Docencia/Visión con IA/6. Teledetección con GEE/Ejemplo Análisis Valle Aburrá/')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Google Earth Engine (GEE)

In [ ]:
#IMPORTANT!
#1. USE COLAB WITH A GOOGLE PERSONAL ACCOUNT (NOT @elpoli.edu.co)
#2. SIGN IN TO https://developers.google.com/earth-engine
#3. CREATE A PROJECT, IN THIS CASE I NAMED IT "rubenchov". USE YOUR OWN.

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='rubenchov') #UPDATE ACCORDING TO THE NAME YOU CHOSE

# Functions for Optical (RGB and NDVI)

In [ ]:
def mask_s2_clouds(image):
  #https://philippgaertner.github.io/2020/08/percent-cloud-cover/
  qa = image.select('QA60')

  # Bits 10 and 11 are clouds and cirrus, respectively.
  cloud_bit_mask = 1 << 10
  cirrus_bit_mask = 1 << 11

  # Both flags should be set to zero, indicating clear conditions.
  mask = (
      qa.bitwiseAnd(cloud_bit_mask)
      .eq(0)
      .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
  )
  return image.updateMask(mask).divide(10000)

def look_opt(LON1, LAT1, LON2, LAT2, date1, date2, clouds):
  ROI = ee.Geometry.Rectangle(LON1, LAT1, LON2, LAT2)

  dataset_s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterDate(date1, date2)
              .filterBounds(ROI)
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', clouds))
              .map(mask_s2_clouds))

  bandas = ['B4', 'B3', 'B2', 'B8'] #B4:Red, B3:Green, B2:Blue, B8:NIR #https://www.satimagingcorp.com/satellite-sensors/other-satellite-sensors/sentinel-2a/
  visualization_s2 = {'min': 0,
                    'max': 0.3,
                    'bands': bandas[0:3]}
  imagen = dataset_s2.select(bandas).median().clip(ROI)
  imagen = imagen.visualize(**visualization_s2)

  imagen_median = dataset_s2.median().clip(ROI)
  ndvi = imagen_median.normalizedDifference(['B8', 'B4']).rename('NDVI')
  ndvi = ndvi.visualize(min = -1, max = 1)

  return imagen, ndvi

In [ ]:
#lat1, long1, lat2, long2 = 5.35859, -72.41807, 5.31723, -72.37142 #Yopal
#subfolder = 'DownloadedYopal'

#Laureles
lat1, long1, lat2, long2 = 6.250722592960795, -75.60232402704338, 6.2319517640375, -75.57494403845106 #Valle Aburrá Laureles
#subfolder = 'DownloadedLaureles'

#Todo el Valle de Aburrá
lat1, long1, lat2, long2 = 6.3596278, -75.64220277777778, 6.1201472, -75.53566111111111 #Todo el Valle de Aburrá
subfolder = 'DownloadedAburra'

years = ['2025']
#years = ['2018', '2019', '2020', '2021', '2022', '2023', '2024']


subfolder_path = os.path.join(dirname, subfolder)
if not os.path.exists(subfolder_path):
  os.makedirs(subfolder_path)

In [ ]:
for year in years:
  print(year)
  date1 = year + '-01-01'
  date2 = year + '-12-31'
  img, ndvi = look_opt(long1, lat1, long2, lat2, date1, date2, 40)

  #Export RGB
  export_path = os.path.join(subfolder_path, year + '-RGB.tif')
  geemap.ee_export_image(img, export_path, scale = 10, file_per_band=False)

  #Export NDVI
  export_path = os.path.join(subfolder_path, year + '-NDVI.tif')
  geemap.ee_export_image(ndvi, export_path, scale = 10, file_per_band=False)

2025
Generating URL ...
Please wait ...
Data downloaded to /content/drive/MyDrive/2026/Docencia/Visión con IA/6. Teledetección con GEE/Ejemplo Análisis Valle Aburrá/DownloadedAburra/2025-RGB.tif
Generating URL ...
Please wait ...
Data downloaded to /content/drive/MyDrive/2026/Docencia/Visión con IA/6. Teledetección con GEE/Ejemplo Análisis Valle Aburrá/DownloadedAburra/2025-NDVI.tif


# (Optional) More with radar (SAR)

Functions to download images

In [ ]:
def look_SAR(LON1, LAT1, LON2, LAT2, date1, date2, pol = 'VH', dir = 'DESCENDING'):
  p11 = ee.Geometry.Point(LON1, LAT1)
  p12 = ee.Geometry.Point(LON1, LAT2)
  p21 = ee.Geometry.Point(LON2, LAT1)
  p22 = ee.Geometry.Point(LON2, LAT2)

  sar = (ee.ImageCollection('COPERNICUS/S1_GRD').
        filter(ee.Filter.listContains('transmitterReceiverPolarisation', pol)).
        filterBounds(p11).filterBounds(p12).filterBounds(p21).filterBounds(p22).filterDate(date1, date2).
        filter(ee.Filter.eq('instrumentMode', 'IW')).
        filter(ee.Filter.eq('orbitProperties_pass', dir)).
        select(pol))

  ROI = ee.Geometry.Rectangle(LON1, LAT1, LON2, LAT2)
  print('Images found: ', sar.size().getInfo())
  if sar.size().getInfo() == 0:
    return None
  else:
    return sar.first().clip(ROI)

Export images

In [ ]:
months = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
for year in years:
  print(year)
  for month in months:
    first_day = '01'
    if month == '02':
      last_day = '28'
    else:
      last_day = '30'
    start_date = year + '-' + month + '-' + first_day
    end_date = year + '-' + month + '-' + last_day

    #Export SAR
    pol = 'VH'
    sar = look_SAR(long1, lat1, long2, lat2, start_date, end_date, pol = pol, dir = 'ASCENDING')
    if sar is not None:
      sar_vis = sar.visualize(**{'min' : -25, 'max' : 5})
      export_path = os.path.join(subfolder_path, year + '-' + month + '-' + 'SAR.tif')
      geemap.ee_export_image(sar_vis, export_path, scale = 10, file_per_band=False)

2025
Images found:  0
Images found:  0
Images found:  0
Images found:  2
Generating URL ...
Please wait ...
Data downloaded to /content/drive/MyDrive/2026/Docencia/Visión e IA/6. Teledetección con GEE/Ejemplo Análisis Valle Aburrá/DownloadedLaureles/2025-04-SAR.tif
Images found:  1
Generating URL ...
Please wait ...
Data downloaded to /content/drive/MyDrive/2026/Docencia/Visión e IA/6. Teledetección con GEE/Ejemplo Análisis Valle Aburrá/DownloadedLaureles/2025-05-SAR.tif
Images found:  4
Generating URL ...
Please wait ...
Data downloaded to /content/drive/MyDrive/2026/Docencia/Visión e IA/6. Teledetección con GEE/Ejemplo Análisis Valle Aburrá/DownloadedLaureles/2025-06-SAR.tif
Images found:  2
Generating URL ...
Please wait ...
Data downloaded to /content/drive/MyDrive/2026/Docencia/Visión e IA/6. Teledetección con GEE/Ejemplo Análisis Valle Aburrá/DownloadedLaureles/2025-07-SAR.tif
Images found:  2
Generating URL ...
Please wait ...
Data downloaded to /content/drive/MyDrive/2026/Docen